In [9]:
import pandas as pd

results = pd.read_csv("metrics_results/bus_ontime_results.csv")

for rt in ["60", "77"]:
    print(f"\n{'='*60}")
    print(f"  ROUTE {rt}")
    print(f"{'='*60}")
    
    df = results[results["rt"] == rt].copy()
    print(f"Total observations: {len(df)}")
    print(f"Status breakdown:\n{df['on_time_status'].value_counts().to_string()}")
    print()
    
    # Where dly disagrees with us
    df["dly_says_late"] = df["dly"].astype(bool)
    df["we_say_late"] = df["on_time_status"] == "late"
    disagree = df[df["dly_says_late"] != df["we_say_late"]]
    print(f"Disagreements with dly ({len(disagree)} rows):")
    print(disagree[["vid","tmstmp","pdist","delay_minutes","on_time_status",
                     "dly","matched_trip_id","prev_stop_name","next_stop_name",
                     "time_match_score_minutes"]].to_string())
    print()

    # Full picture of all vehicles on these routes
    print("All observations:")
    print(df[["vid","tmstmp","pdist","delay_minutes","on_time_status",
              "dly","time_match_score_minutes","prev_stop_name","next_stop_name"]
             ].sort_values(["vid","tmstmp"]).to_string())


  ROUTE 60
Total observations: 10
Status breakdown:
on_time_status
on_time    7
early      3

Disagreements with dly (2 rows):
      vid               tmstmp  pdist  delay_minutes on_time_status   dly  matched_trip_id        prev_stop_name          next_stop_name  time_match_score_minutes
331  8279  2026-02-19 23:38:00  49661           -1.6          early  True    6770005108020  79th Street & Hamlin  79th Street & Lawndale                      0.05
332  8279  2026-02-19 23:40:00  49661            1.6        on_time  True    6770002395070   79th Street & Tripp   79th Street & Kostner                      1.11

All observations:
      vid               tmstmp  pdist  delay_minutes on_time_status    dly  time_match_score_minutes            prev_stop_name                               next_stop_name
331  8279  2026-02-19 23:38:00  49661           -1.6          early   True                      0.05      79th Street & Hamlin                       79th Street & Lawndale
332  8279  2026-02-1

In [10]:
import pandas as pd

vehicles = pd.read_csv("data/bus_data_current_chicago.csv")
vehicles["tmstmp"] = pd.to_datetime(vehicles["tmstmp"])
vehicles = vehicles[vehicles["vid"].astype(str) == "8279"].sort_values("tmstmp")
print(vehicles[["vid","tmstmp","pdist","rt"]].to_string())
print()
print("pdist delta:", vehicles["pdist"].diff().abs().values)
print("time delta (min):", vehicles["tmstmp"].diff().dt.total_seconds().div(60).values)

      vid              tmstmp  pdist  rt
250  8279 2026-02-19 23:38:00  49661  60
538  8279 2026-02-19 23:40:00  49661  60

pdist delta: [nan  0.]
time delta (min): [nan  2.]


In [11]:
import pandas as pd

vehicles["tmstmp"] = pd.to_datetime(vehicles["tmstmp"])

print("vid dtype before cleaning:", vehicles["vid"].dtype)
print("sample vids:", vehicles["vid"].head(5).tolist())

# Simulate exactly what detect_ghost_buses does
vehicles["vid"] = vehicles["vid"].astype(str)
vehicles = vehicles.sort_values(["vid", "tmstmp"])

g = vehicles.groupby("vid")
vehicles["_prev_pdist"] = g["pdist"].shift(1)
vehicles["_prev_tmstmp"] = g["tmstmp"].shift(1)
vehicles["_pdist_delta"] = (vehicles["pdist"] - vehicles["_prev_pdist"]).abs()
vehicles["_time_delta_min"] = (vehicles["tmstmp"] - vehicles["_prev_tmstmp"]).dt.total_seconds() / 60

# Check specifically for vid 8279
v = vehicles[vehicles["vid"] == "8279"]
print()
print("vid 8279 ghost signals:")
print(v[["vid","tmstmp","pdist","_prev_pdist","_pdist_delta","_time_delta_min"]].to_string())
print()

# Check the ghost conditions
v = v.copy()
v["ghost_frozen"] = (
    (v["_pdist_delta"] < 50) &
    (v["_time_delta_min"] > 1) &
    v["_prev_pdist"].notna()
)
print("ghost_frozen:", v["ghost_frozen"].tolist())
print("ghost_score:", (v["ghost_frozen"].astype(int) + 
                       ((v["_pdist_delta"] > 5280) & v["_prev_pdist"].notna()).astype(int)).tolist())

vid dtype before cleaning: int64
sample vids: [8279, 8279]

vid 8279 ghost signals:
      vid              tmstmp  pdist  _prev_pdist  _pdist_delta  _time_delta_min
250  8279 2026-02-19 23:38:00  49661          NaN           NaN              NaN
538  8279 2026-02-19 23:40:00  49661      49661.0           0.0              2.0

ghost_frozen: [False, True]
ghost_score: [0, 1]


In [13]:
import pandas as pd

results = pd.read_csv("bus_ontime_results.csv", low_memory=False)

rt19 = results[results["rt"] == "19"].copy()
rt19["dly_says_late"] = rt19["dly"].astype(bool)
rt19["we_say_late"]   = rt19["on_time_status"] == "late"
rt19["agree"]         = rt19["dly_says_late"] == rt19["we_say_late"]

print(f"Route 19 total observations: {len(rt19)}")
print(f"Status breakdown:\n{rt19['on_time_status'].value_counts().to_string()}")
print()
print(f"dly breakdown:\n{rt19['dly'].value_counts().to_string()}")
print()

# Focus on disagreements
disagree = rt19[~rt19["agree"]]
print(f"Disagreements: {len(disagree)}")
print()
print("Disagreement breakdown (dly vs our status):")
print(disagree.groupby(["dly", "on_time_status"]).size().to_string())
print()
print("Sample disagreements:")
print(disagree[["vid","tmstmp","pdist","delay_minutes","on_time_status",
                "dly","matched_trip_id","time_match_score_minutes",
                "prev_stop_name","next_stop_name"]].head(20).to_string())

Route 19 total observations: 1877
Status breakdown:
on_time_status
on_time      1533
early         322
late           12
unmatched       6
ghost           4

dly breakdown:
dly
False    940
True     937

Disagreements: 939

Disagreement breakdown (dly vs our status):
dly    on_time_status
False  late                7
True   early             151
       on_time           780
       unmatched           1

Sample disagreements:
          vid               tmstmp  pdist  delay_minutes on_time_status   dly  matched_trip_id  time_match_score_minutes                      prev_stop_name             next_stop_name
1522633  4012  2026-02-24 21:35:00    748           -2.4          early  True     6.770049e+12                      2.37                  Archer & St. Louis             Archer & Homan
1522634  4012  2026-02-24 21:37:00    748            0.1        on_time  True     6.770031e+12                      0.12  Archer & Pope John Paul II/43rd St            Archer & Kedzie
1522635  4012  2026

In [14]:
import pandas as pd

results = pd.read_csv("bus_ontime_results.csv", low_memory=False)

# Check jump distribution — are these concentrated on specific vehicles/routes?
jumps = results[results["ghost_jump"] == True]
print(f"Total jump observations: {len(jumps)}")
print(f"Unique vehicles with jumps: {jumps['vid'].nunique()}")
print()
print("Jumps by route (top 15):")
print(jumps.groupby("rt").size().sort_values(ascending=False).head(15).to_string())
print()
print("Vehicles with most jumps:")
print(jumps.groupby("vid").size().sort_values(ascending=False).head(15).to_string())
print()

# For route 169 specifically
rt169 = results[results["rt"] == "169"].copy()
rt169["dly_says_late"] = rt169["dly"].astype(bool)
rt169["we_say_late"] = rt169["on_time_status"] == "late"
rt169["agree"] = rt169["dly_says_late"] == rt169["we_say_late"]

print(f"\nRoute 169 — {len(rt169)} observations")
print(f"Status: {rt169['on_time_status'].value_counts().to_string()}")
print(f"\ndly breakdown: {rt169['dly'].value_counts().to_string()}")
print(f"\nDisagreement breakdown:")
disagree = rt169[~rt169["agree"]]
print(disagree.groupby(["dly", "on_time_status"]).size().to_string())
print()
print("Sample disagreements:")
print(disagree[["vid","tmstmp","pdist","delay_minutes","on_time_status",
                "dly","time_match_score_minutes","pdist_unreliable",
                "prev_stop_name","next_stop_name"]].head(15).to_string())

Total jump observations: 29746
Unique vehicles with jumps: 1598

Jumps by route (top 15):
rt
J14    6014
147    4778
6      3857
146    2053
47     1977
67     1515
26     1381
28      841
2       748
44      384
148     260
54      252
9       243
136     221
135     216

Vehicles with most jumps:
vid
4324    294
4331    271
4326    264
4011    258
4043    236
4068    235
4083    234
4176    232
4332    232
4070    221
4184    219
4309    208
4304    205
4053    205
4321    204


Route 169 — 1659 observations
Status: on_time_status
on_time    934
late       477
early      247
ghost        1

dly breakdown: dly
False    1643
True       16

Disagreement breakdown:
dly    on_time_status
False  late              466
True   early               1
       on_time             4

Sample disagreements:
          vid               tmstmp  pdist  delay_minutes on_time_status    dly  time_match_score_minutes  pdist_unreliable                  prev_stop_name           next_stop_name
1593704  4074  2

In [15]:
import pandas as pd

trips = pd.read_csv("Datasets/trips.txt", dtype=str)
rt169 = trips[trips["route_id"] == "169"]
print(f"Route 169 trips: {len(rt169)}")
print(f"Direction values: {rt169['direction_id'].value_counts().to_string()}")
print(f"Direction (text): {rt169['direction'].value_counts().to_string()}")
print()

# Also check if pdist values on route 169 are consistent
results = pd.read_csv("bus_ontime_results.csv", low_memory=False)
r169 = results[results["rt"] == "169"]
print(f"pdist range on route 169: {r169['pdist'].min()} - {r169['pdist'].max()}")
print(f"pdist std: {r169['pdist'].std():.0f}")
print(f"Unique vehicles: {r169['vid'].nunique()}")
print()

# Sample a vehicle and show its full pdist trajectory
sample_vid = r169["vid"].value_counts().index[0]
v = r169[r169["vid"] == sample_vid].sort_values("tmstmp")
print(f"Sample vehicle {sample_vid} on route 169:")
print(v[["tmstmp","pdist","delay_minutes","on_time_status","dly",
         "time_match_score_minutes"]].head(20).to_string())

Route 169 trips: 11
Direction values: direction_id
0    6
1    5
Direction (text): direction
West    6
East    5

pdist range on route 169: 0 - 88677
pdist std: 26588
Unique vehicles: 17

Sample vehicle 4332 on route 169:
                      tmstmp  pdist  delay_minutes on_time_status    dly  time_match_score_minutes
1758406  2026-02-20 20:51:00      0           -2.0          early  False                      2.00
1758407  2026-02-20 20:52:00      0           -1.0        on_time  False                      1.00
1758408  2026-02-20 20:54:00    525            0.6        on_time  False                      0.62
1758409  2026-02-20 20:55:00    759            1.4        on_time  False                      1.45
1758410  2026-02-20 20:57:00   1558            2.9        on_time  False                      2.86
1758411  2026-02-20 21:00:00   5718            2.7        on_time  False                      2.69
1758412  2026-02-20 21:02:00   7852            3.0        on_time  False             

In [16]:
import pandas as pd

# trips  = pd.read_csv("trips.txt", dtype=str)
st     = pd.read_parquet("gtfs_cache/stop_times_enriched.parquet")
results = pd.read_csv("bus_ontime_results.csv", low_memory=False)

# What times does route 169 actually operate according to GTFS?
rt169_trips = trips[trips["route_id"] == "169"]["trip_id"].tolist()
rt169_st    = st[st["trip_id"].isin(rt169_trips)]

print("Route 169 scheduled time windows:")
trip_windows = (
    rt169_st.groupby("trip_id")["arrival_minutes"]
    .agg(["min", "max"])
    .assign(
        start_time=lambda d: d["min"].apply(lambda m: f"{int(m//60):02d}:{int(m%60):02d}"),
        end_time  =lambda d: d["max"].apply(lambda m: f"{int(m//60):02d}:{int(m%60):02d}"),
    )
    .merge(trips[["trip_id","direction"]], on="trip_id")
    .sort_values("min")
)
print(trip_windows[["trip_id","direction","start_time","end_time"]].to_string())
print()

# What times are route 169 vehicles actually observed?
r169_results = results[results["rt"] == "169"].copy()
r169_results["tmstmp"] = pd.to_datetime(r169_results["tmstmp"])
r169_results["hour"]   = r169_results["tmstmp"].dt.hour

print("Route 169 vehicle observations by hour:")
print(r169_results.groupby("hour").agg(
    obs=("vid","count"),
    avg_delay=("delay_minutes","mean"),
    pct_late=("on_time_status", lambda x: (x=="late").mean()*100),
    avg_match_score=("time_match_score_minutes","mean")
).round(1).to_string())

Route 169 scheduled time windows:
          trip_id direction start_time end_time
6   6770033432020      West      02:34    03:20
8   6770048612010      West      02:34    03:20
0   6770001509020      East      03:40    04:22
4   6770018685020      West      08:27    09:20
1   6770003339020      West      08:32    09:25
7   6770043866020      East      09:45    10:32
9   6770050008020      East      09:50    10:37
2   6770004319020      West      15:15    16:20
10  6770050121020      East      15:15    16:14
5   6770030005020      West      20:53    21:45
3   6770012465020      East      21:50    22:32

Route 169 vehicle observations by hour:
      obs  avg_delay  pct_late  avg_match_score
hour                                           
2      80        2.4      18.8              3.1
3     165        3.2      38.8              6.4
4      72        1.2      30.6              4.5
8     131        2.1      10.7              2.9
9     204        5.4      42.6              6.4
10    235    

In [17]:
import pandas as pd

results = pd.read_csv("bus_ontime_results.csv", low_memory=False)

jumps = results[results["ghost_jump"] == True]
print(f"Jump observations: {len(jumps)}")
print(f"Are any of these classified as ghost? {jumps['is_ghost'].sum()}")
print()

# Sample jump vehicles to see what their pdist looks like
sample_vid = jumps["vid"].value_counts().index[0]
veh = results[results["vid"] == sample_vid].sort_values("tmstmp")
print(f"Sample jump vehicle {sample_vid}:")
print(veh[["tmstmp","rt","pdist","ghost_frozen","ghost_jump","is_ghost","on_time_status"]].head(30).to_string())

Jump observations: 29083
Are any of these classified as ghost? 0

Sample jump vehicle 4324:
                      tmstmp   rt  pdist  ghost_frozen  ghost_jump  is_ghost on_time_status
1741812  2026-02-20 18:41:00  J14    975         False       False     False        on_time
1741813  2026-02-20 18:42:00  J14   1249         False       False     False        on_time
1741814  2026-02-20 18:44:00  J14   2924         False       False     False        on_time
1741815  2026-02-20 18:45:00  J14   3459         False       False     False        on_time
1741816  2026-02-20 18:47:00  J14   4257         False       False     False        on_time
1741817  2026-02-20 18:49:00  J14   4951         False       False     False        on_time
1741818  2026-02-20 18:50:00  J14   5190         False       False     False        on_time
1741819  2026-02-20 18:52:00  J14   6212         False       False     False        on_time
1741820  2026-02-20 18:53:00  J14   6997         False       False     False    